# XAUUSD XGBoost Training

Trains a **42-feature XGBoost model** with regime indicators, US10Y yields, and real DXY.
Outputs a dict artifact compatible with the live inference pipeline (`ModelInference`).

### Two paths:
| Path | When to use | What you need |
|------|-------------|---------------|
| **A — Auto-export** | First time, or fresh data | Twelve Data API key ([free signup](https://twelvedata.com/apikey)) |
| **B — Upload zip** | You already have `data/training/` locally | Zip file |

**Runtime:** ~30-60 min CPU (50 Optuna trials). No GPU needed.

In [ ]:
# @title 0. GitHub username (edit this before running)
GITHUB_USER = "webkaave"  # @param {type:"string"}
REPO_NAME = "tradebot"

In [ ]:
# @title 1. Clone Repo
import os
if os.path.exists(f'/content/{REPO_NAME}'):
    %cd /content/{REPO_NAME}
    !git pull
else:
    %cd /content
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_NAME}

In [ ]:
# @title 2. Install Dependencies
!pip install -e ".[train]" -q
print("Dependencies installed")

In [ ]:
# @title 3. Set Twelve Data API Key
# Option A: Colab Secrets (left sidebar → 🔑 key icon)
#   Name: TWELVE_DATA_API_KEY   Value: your key
# Option B: Type it in below
from google.colab import userdata
import os

try:
    os.environ["TWELVE_DATA_API_KEY"] = userdata.get("TWELVE_DATA_API_KEY")
    print("✅ Loaded from Colab Secrets")
except userdata.SecretNotFoundError:
    key = input("Paste your Twelve Data API key: ")
    os.environ["TWELVE_DATA_API_KEY"] = key

assert os.getenv("TWELVE_DATA_API_KEY"), "API key required"
print(f"Key: {os.environ['TWELVE_DATA_API_KEY'][:8]}...")

In [ ]:
# @title 4. Choose Data Source
data_source = "export"  # @param ["export", "upload"]

if data_source == "upload":
    from google.colab import files
    print("Upload your training_bundle.zip (exported from local machine)")
    uploaded = files.upload()
    !mkdir -p data/training
    !unzip -o training_bundle.zip -d data/training/

print("\nFiles in data/training/:")
!ls -lh data/training/

In [ ]:
# @title 5. Export Data (auto-export path only)
if data_source == "export":
    print("Exporting XAUUSD M15/H1/H4 + DXY M15 + EUR/USD M15...")
    !python scripts/export_training_bundle.py --years 5 --output-dir data/training

    print("\nExporting US10Y yields (no API key needed)...")
    !python scripts/export_fred_series.py --output data/training/us10y_daily.csv

    print("\nFinal file listing:")
    !ls -lh data/training/

In [ ]:
# @title 6. Validate CSVs
!python scripts/validate_training_data.py --csv data/training/xauusd_m15.csv
!python scripts/validate_training_data.py --csv data/training/xauusd_h1.csv
!python scripts/validate_training_data.py --csv data/training/xauusd_h4.csv
!python scripts/validate_training_data.py --csv data/training/eurusd_m15.csv
!python scripts/validate_training_data.py --csv data/training/dxy_m15.csv
!python scripts/validate_training_data.py --csv data/training/us10y_daily.csv

In [ ]:
# @title 7. Sanitize (fix high/low inversions)
import glob
for f in sorted(glob.glob('data/training/*.csv')):
    !python scripts/sanitize_training_data.py "$f"

In [ ]:
# @title 8. Train
# Edit these if you want to change the training config
TRIALS = 50
TARGET_MODE = "close_return"   # close_return | first_touch | three_class | binary
LOOKAHEAD = 8
ATR_THRESHOLD = 1.0
BINARY = True                  # True = overlap_macro_trend artifact (BUY-focused)

!python scripts/train_xgboost.py \
  --m15 data/training/xauusd_m15.csv \
  --h1 data/training/xauusd_h1.csv \
  --h4 data/training/xauusd_h4.csv \
  --dxy data/training/eurusd_m15.csv \
  --us10y data/training/us10y_daily.csv \
  --real-dxy data/training/dxy_m15.csv \
  --output models/xgb_xauusd_v1.pkl \
  --trials {TRIALS} \
  --target-mode {TARGET_MODE} \
  --lookahead {LOOKAHEAD} \
  --atr-threshold {ATR_THRESHOLD} \
  --binary

In [ ]:
# @title 9. Inspect Artifact
import joblib
artifact = joblib.load('models/xgb_xauusd_v1.pkl')
print(f"artifact_type:    {artifact.get('artifact_type')}")
print(f"feature_columns:  {len(artifact.get('feature_columns', []))}")
print(f"threshold:        {artifact.get('threshold')}")
print(f"enabled_sides:    {artifact.get('enabled_sides')}")
model = artifact.get('model') or artifact
print(f"model type:       {type(model).__name__}")

In [ ]:
# @title 10. Download
from google.colab import files
files.download('models/xgb_xauusd_v1.pkl')

---
## After download — back on your local machine

```bash
# 1. Place the model
mv ~/Downloads/xgb_xauusd_v1.pkl models/overlap_macro_trend_xgb.pkl

# 2. Verify
python -c "import joblib; a=joblib.load('models/overlap_macro_trend_xgb.pkl'); print('OK,', a.get('artifact_type'))"

# 3. Test with a dry signal
python scripts/dry_model_signal.py --ignore-calendar
```